 # 03 · Data Quality Silver — YelpConnect

 **Tipo:** Data Quality / Quality Gate  
 **Capa evaluada:** Silver  
 **Catálogo de reglas:** `workspace.yelp_gov.dq_reglas`  
 **Histórico de resultados:** `workspace.yelp_gov.dq_resultados`

 ---

 ## Objetivo

 Ejecutar las reglas de calidad publicadas para la capa Silver,
 registrar evidencia histórica de cada evaluación y determinar
 si los datos pueden continuar hacia Trust & Safety.

 ## Principio

 **Silver conforma · DQ valida · Trust & Safety evalúa señales**

 Este notebook:

 - valida la infraestructura requerida;
 - valida los contratos de gobierno;
 - recupera las reglas `DQ-SL-*`;
 - ejecuta el motor genérico de Data Quality;
 - calcula métricas de cumplimiento;
 - clasifica resultados como `PASS`, `WARNING` o `FAIL`;
 - persiste los resultados históricamente;
 - genera scorecards;
 - aplica el Quality Gate Silver.

 Este notebook **NO**:

 - crea schemas;
 - crea tablas de gobierno;
 - publica reglas DQ;
 - transforma datos Silver;
 - elimina registros;
 - corrige valores;
 - ejecuta reglas Trust & Safety.

 ---

 ## Política del Quality Gate

 - `PASS`: la regla alcanza su umbral y no presenta incumplimientos.
 - `WARNING`: existen incumplimientos, pero se alcanza el umbral definido.
 - `FAIL`: el porcentaje de cumplimiento es inferior al umbral.
 - Solo un `FAIL` con criticidad **alta** bloquea el pipeline.

 Los `WARNING` y los `FAIL` no críticos permanecen registrados
 como evidencia, pero no detienen automáticamente el flujo.


In [0]:
# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import uuid

from datetime import (
    datetime,
    timezone
)

from pyspark.sql import functions as F


In [0]:
# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOV_SCHEMA = "yelp_gov"

SILVER = (
    f"{CATALOG}."
    f"{SILVER_SCHEMA}"
)

LAYER = "silver"

PIPELINE_NAME = (
    "E5_03_dq_silver"
)

DQ_RULE_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_reglas"
)

DQ_RESULT_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_resultados"
)

DQ_EXECUTION_ID = str(
    uuid.uuid4()
)

DQ_EXECUTION_TS = (
    datetime.now(
        timezone.utc
    )
)

EXPECTED_RULE_COUNT = 22

EXPECTED_RULE_IDS = {
    f"DQ-SL-{number:03d}"
    for number
    in range(
        1,
        EXPECTED_RULE_COUNT + 1
    )
}

SUPPORTED_RULE_TYPES = {
    "TABLE_NOT_EMPTY",
    "CONDITION",
    "UNIQUENESS"
}

SUPPORTED_CRITICALITIES = {
    "alta",
    "media",
    "baja"
}


print("=" * 72)
print("YELPCONNECT - DATA QUALITY SILVER")
print("=" * 72)

print(
    f"Pipeline      : {PIPELINE_NAME}"
)

print(
    f"Execution ID  : {DQ_EXECUTION_ID}"
)

print(
    f"Execution UTC : "
    f"{DQ_EXECUTION_TS.isoformat()}"
)

print(
    f"Layer         : {LAYER}"
)

print(
    f"Reglas        : {DQ_RULE_TABLE}"
)

print(
    f"Resultados    : {DQ_RESULT_TABLE}"
)

print("=" * 72)


# COMMAND ----------

 # 1 · Precondiciones

 Data Quality Silver depende de infraestructura creada previamente.

 Antes de ejecutar reglas se valida:

 1. existencia del schema Silver;
 2. existencia de `dq_reglas`;
 3. existencia de `dq_resultados`;
 4. compatibilidad de los contratos físicos.

 Este notebook no intenta reparar contratos automáticamente.

 Si existe una inconsistencia estructural, la ejecución debe fallar
 antes de evaluar cualquier regla.

In [0]:

# COMMAND ----------

# ============================================================
# 3. VALIDAR SCHEMA SILVER
# ============================================================

try:

    spark.sql(
        f"""
        DESCRIBE SCHEMA
        {SILVER}
        """
    )

except Exception as exc:

    raise RuntimeError(
        f"""
        ❌ SCHEMA SILVER INEXISTENTE

        Schema:
        {SILVER}

        Ejecutar previamente:
        00_setup/01_configurar_lakehouse
        """
    ) from exc


print(
    f"✔ Schema Silver disponible: "
    f"{SILVER}"
)



In [0]:
# COMMAND ----------

# ============================================================
# 4. FUNCIÓN PARA VALIDAR CONTRATOS
# ============================================================

def assert_table_contract(
    table_name,
    expected_contract
):

    if not spark.catalog.tableExists(
        table_name
    ):

        raise RuntimeError(
            f"""
            ❌ TABLA REQUERIDA INEXISTENTE

            Tabla:
            {table_name}
            """
        )


    actual_contract = {

        field.name:
            field.dataType
            .simpleString()
            .lower()

        for field
        in spark.table(
            table_name
        ).schema.fields
    }


    missing_columns = [

        column_name

        for column_name
        in expected_contract

        if column_name
        not in actual_contract
    ]


    if missing_columns:

        raise RuntimeError(
            f"""
            ❌ CONTRATO INCOMPLETO

            Tabla:
            {table_name}

            Columnas faltantes:
            {missing_columns}
            """
        )


    invalid_types = []


    for (
        column_name,
        expected_type
    ) in expected_contract.items():

        actual_type = (
            actual_contract[
                column_name
            ]
        )

        if (
            actual_type
            !=
            expected_type.lower()
        ):

            invalid_types.append(
                {
                    "column":
                        column_name,

                    "expected":
                        expected_type,

                    "actual":
                        actual_type
                }
            )


    if invalid_types:

        raise RuntimeError(
            f"""
            ❌ CONTRATO CON TIPOS INCOMPATIBLES

            Tabla:
            {table_name}

            Detalle:
            {invalid_types}
            """
        )


    print(
        f"✔ Contrato válido: "
        f"{table_name}"
    )

In [0]:

# COMMAND ----------

# ============================================================
# 5. CONTRATO dq_reglas
# ============================================================

DQ_RULE_CONTRACT = {

    "regla_id":
        "string",

    "capa":
        "string",

    "tabla_objetivo":
        "string",

    "columna":
        "string",

    "dimension":
        "string",

    "tipo_regla":
        "string",

    "descripcion":
        "string",

    "expresion_sql":
        "string",

    "criticidad":
        "string",

    "umbral_pct":
        "double"
}


assert_table_contract(
    DQ_RULE_TABLE,
    DQ_RULE_CONTRACT
)


In [0]:
# COMMAND ----------

# ============================================================
# 6. CONTRATO dq_resultados
# ============================================================

DQ_RESULT_CONTRACT = {

    "ejecucion_id":
        "string",

    "ejecucion_ts":
        "timestamp",

    "regla_id":
        "string",

    "capa":
        "string",

    "tabla_objetivo":
        "string",

    "columna":
        "string",

    "dimension":
        "string",

    "tipo_regla":
        "string",

    "criticidad":
        "string",

    "filas_totales":
        "bigint",

    "filas_ok":
        "bigint",

    "filas_error":
        "bigint",

    "pct_cumplimiento":
        "double",

    "umbral_pct":
        "double",

    "paso":
        "boolean"
}


assert_table_contract(
    DQ_RESULT_TABLE,
    DQ_RESULT_CONTRACT
)


# COMMAND ----------

 # 2 · Catálogo de reglas Silver

 Las reglas no están implementadas dentro del motor.

 Se recuperan dinámicamente desde:

 `workspace.yelp_gov.dq_reglas`

 y deben haber sido publicadas previamente mediante:

 `01_publicar_catalogo_reglas_dq_silver`

 Para esta versión del modelo se esperan exactamente:

 `DQ-SL-001 ... DQ-SL-022`

 Esto evita que el pipeline continúe silenciosamente con un catálogo
 incompleto o con reglas no controladas.

In [0]:
# COMMAND ----------

# ============================================================
# 7. CARGAR REGLAS SILVER
# ============================================================

df_rules = (

    spark

    .table(
        DQ_RULE_TABLE
    )

    .filter(
        F.lower(
            F.trim(
                F.col(
                    "capa"
                )
            )
        )
        ==
        LAYER
    )

    .orderBy(
        "regla_id"
    )
)


actual_rule_count = (
    df_rules.count()
)


if (
    actual_rule_count
    !=
    EXPECTED_RULE_COUNT
):

    raise RuntimeError(
        f"""
        ❌ CANTIDAD DE REGLAS SILVER INCORRECTA

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Encontradas:
        {actual_rule_count}

        Ejecutar previamente:
        E5_calidad_datos/
        01_publicar_catalogo_reglas_dq_silver
        """
    )


print(
    f"✔ Reglas Silver cargadas: "
    f"{actual_rule_count}"
)

In [0]:

# COMMAND ----------

# ============================================================
# 8. VALIDAR IDS CANÓNICOS
# ============================================================

actual_rule_ids = {

    row.regla_id

    for row
    in df_rules
        .select(
            "regla_id"
        )
        .collect()
}


missing_rule_ids = (
    EXPECTED_RULE_IDS
    -
    actual_rule_ids
)


extra_rule_ids = (
    actual_rule_ids
    -
    EXPECTED_RULE_IDS
)


if (
    missing_rule_ids
    or
    extra_rule_ids
):

    raise RuntimeError(
        f"""
        ❌ CATÁLOGO SILVER NO CANÓNICO

        IDs faltantes:
        {sorted(missing_rule_ids)}

        IDs adicionales:
        {sorted(extra_rule_ids)}
        """
    )


print(
    "✔ IDs DQ-SL-001 ... "
    "DQ-SL-022 validados."
)


In [0]:

# COMMAND ----------

# ============================================================
# 9. VALIDAR DUPLICADOS DE regla_id
# ============================================================

duplicate_rule_ids = (

    df_rules

    .groupBy(
        "regla_id"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        >
        1
    )

    .collect()
)


if duplicate_rule_ids:

    raise RuntimeError(
        f"""
        ❌ EXISTEN regla_id DUPLICADOS

        {duplicate_rule_ids}
        """
    )


print(
    "✔ No existen regla_id duplicados."
)


In [0]:

# COMMAND ----------

# ============================================================
# 10. VALIDAR TIPOS DE REGLA
# ============================================================

invalid_rule_types = (

    df_rules

    .filter(
        ~F.upper(
            F.trim(
                F.col(
                    "tipo_regla"
                )
            )
        ).isin(
            list(
                SUPPORTED_RULE_TYPES
            )
        )
    )

    .select(
        "regla_id",
        "tipo_regla"
    )

    .collect()
)


if invalid_rule_types:

    raise RuntimeError(
        f"""
        ❌ TIPOS DE REGLA NO SOPORTADOS

        {invalid_rule_types}
        """
    )


print(
    "✔ Tipos de regla soportados."
)


In [0]:
# COMMAND ----------

# ============================================================
# 11. VALIDAR CRITICIDAD
# ============================================================

invalid_criticalities = (

    df_rules

    .filter(
        ~F.lower(
            F.trim(
                F.col(
                    "criticidad"
                )
            )
        ).isin(
            list(
                SUPPORTED_CRITICALITIES
            )
        )
    )

    .select(
        "regla_id",
        "criticidad"
    )

    .collect()
)


if invalid_criticalities:

    raise RuntimeError(
        f"""
        ❌ CRITICIDADES NO SOPORTADAS

        {invalid_criticalities}
        """
    )


print(
    "✔ Criticidades válidas."
)

In [0]:
# COMMAND ----------

# ============================================================
# 12. VALIDAR UMBRALES
# ============================================================

invalid_thresholds = (

    df_rules

    .filter(
        F.col(
            "umbral_pct"
        ).isNull()

        |

        (
            F.col(
                "umbral_pct"
            )
            <
            0.0
        )

        |

        (
            F.col(
                "umbral_pct"
            )
            >
            100.0
        )
    )

    .select(
        "regla_id",
        "umbral_pct"
    )

    .collect()
)


if invalid_thresholds:

    raise RuntimeError(
        f"""
        ❌ UMBRALES DQ INVÁLIDOS

        {invalid_thresholds}
        """
    )


print(
    "✔ Umbrales válidos."
)

In [0]:



# COMMAND ----------

# ============================================================
# 13. VALIDAR CONDITION
# ============================================================

invalid_conditions = (

    df_rules

    .filter(
        (
            F.upper(
                F.trim(
                    F.col(
                        "tipo_regla"
                    )
                )
            )
            ==
            "CONDITION"
        )

        &

        (
            F.col(
                "expresion_sql"
            ).isNull()

            |

            (
                F.length(
                    F.trim(
                        F.col(
                            "expresion_sql"
                        )
                    )
                )
                ==
                0
            )
        )
    )

    .select(
        "regla_id"
    )

    .collect()
)


if invalid_conditions:

    raise RuntimeError(
        f"""
        ❌ CONDITION SIN expresion_sql

        {invalid_conditions}
        """
    )


print(
    "✔ Reglas CONDITION válidas."
)


In [0]:


# COMMAND ----------

# ============================================================
# 14. VALIDAR UNIQUENESS
# ============================================================

invalid_uniqueness = (

    df_rules

    .filter(
        (
            F.upper(
                F.trim(
                    F.col(
                        "tipo_regla"
                    )
                )
            )
            ==
            "UNIQUENESS"
        )

        &

        (
            F.col(
                "columna"
            ).isNull()

            |

            (
                F.col(
                    "columna"
                )
                ==
                "*"
            )
        )
    )

    .select(
        "regla_id",
        "columna"
    )

    .collect()
)


if invalid_uniqueness:

    raise RuntimeError(
        f"""
        ❌ UNIQUENESS SIN COLUMNA VÁLIDA

        {invalid_uniqueness}
        """
    )


print(
    "✔ Reglas UNIQUENESS válidas."
)


# COMMAND ----------


 # 3 · Tablas Silver requeridas

 Las tablas que debe evaluar el motor se obtienen directamente
 desde el catálogo.

 Esto evita mantener una segunda lista manual que podría quedar
 desalineada con las reglas.

 Todas las tablas objetivo deben:

 - pertenecer a `workspace.yelp_silver`;
 - existir antes de iniciar la evaluación.


In [0]:

# COMMAND ----------

# ============================================================
# 15. VALIDAR TABLAS OBJETIVO
# ============================================================

target_tables = {

    row.tabla_objetivo

    for row
    in df_rules
        .select(
            "tabla_objetivo"
        )
        .distinct()
        .collect()
}


invalid_target_tables = [

    table_name

    for table_name
    in target_tables

    if not table_name.startswith(
        f"{SILVER}."
    )
]


if invalid_target_tables:

    raise RuntimeError(
        f"""
        ❌ CATÁLOGO SILVER CONTIENE
        TABLAS FUERA DE yelp_silver

        Tablas:

        {invalid_target_tables}
        """
    )


missing_target_tables = [

    table_name

    for table_name
    in target_tables

    if not spark.catalog.tableExists(
        table_name
    )
]


if missing_target_tables:

    raise RuntimeError(
        f"""
        ❌ FALTAN TABLAS SILVER

        Tablas:

        {missing_target_tables}

        Ejecutar previamente:
        E7_pipeline_lakehouse/
        02_silver_transformacion
        """
    )


print(
    f"✔ Tablas objetivo disponibles: "
    f"{len(target_tables)}"
)


# COMMAND ----------


 # 4 · Motor genérico de Data Quality

 El motor soporta tres tipos de regla:

 ### `TABLE_NOT_EMPTY`

 Comprueba que una tabla contenga registros.

 ### `CONDITION`

 Evalúa una expresión SQL booleana almacenada en el catálogo.
 Cada registro que cumple la expresión se considera válido.

 ### `UNIQUENESS`

 Comprueba el nivel de unicidad de una columna.

 ---

 Para cada regla se calcula:

 - filas totales;
 - filas válidas;
 - filas con incumplimiento;
 - porcentaje de cumplimiento;
 - umbral esperado;
 - resultado booleano `paso`.

 El motor no modifica las tablas evaluadas.


In [0]:

# COMMAND ----------

# ============================================================
# 16. EJECUTAR MOTOR DQ SILVER
# ============================================================

rules = (
    df_rules.collect()
)

results = []

table_cache = {}
table_counts = {}


for rule in rules:

    print()
    print("-" * 72)

    print(
        f"Regla       : "
        f"{rule.regla_id}"
    )

    print(
        f"Descripción : "
        f"{rule.descripcion}"
    )

    print(
        f"Tabla       : "
        f"{rule.tabla_objetivo}"
    )

    print(
        f"Columna     : "
        f"{rule.columna}"
    )

    print(
        f"Dimensión   : "
        f"{rule.dimension}"
    )

    print(
        f"Tipo        : "
        f"{rule.tipo_regla}"
    )

    print(
        f"Criticidad  : "
        f"{rule.criticidad}"
    )


    # ========================================================
    # DATAFRAME OBJETIVO
    # ========================================================

    if (
        rule.tabla_objetivo
        not in table_cache
    ):

        table_cache[
            rule.tabla_objetivo
        ] = spark.table(
            rule.tabla_objetivo
        )


    df_target = (
        table_cache[
            rule.tabla_objetivo
        ]
    )


    # ========================================================
    # TOTAL DE REGISTROS
    # ========================================================

    if (
        rule.tabla_objetivo
        not in table_counts
    ):

        table_counts[
            rule.tabla_objetivo
        ] = (
            df_target.count()
        )


    total = (
        table_counts[
            rule.tabla_objetivo
        ]
    )


    tipo = (
        rule.tipo_regla
        .upper()
        .strip()
    )


    # ========================================================
    # TABLE_NOT_EMPTY
    # ========================================================

    if (
        tipo
        ==
        "TABLE_NOT_EMPTY"
    ):

        filas_ok = (
            total
            if total > 0
            else 0
        )


    # ========================================================
    # CONDITION
    # ========================================================

    elif (
        tipo
        ==
        "CONDITION"
    ):

        try:

            filas_ok = (

                df_target

                .filter(
                    F.expr(
                        rule.expresion_sql
                    )
                )

                .count()
            )

        except Exception as exc:

            raise RuntimeError(
                f"""
                ❌ ERROR EJECUTANDO CONDITION

                Regla:
                {rule.regla_id}

                Tabla:
                {rule.tabla_objetivo}

                Expresión:
                {rule.expresion_sql}
                """
            ) from exc


    # ========================================================
    # UNIQUENESS
    # ========================================================

    elif (
        tipo
        ==
        "UNIQUENESS"
    ):

        if (
            rule.columna
            not in df_target.columns
        ):

            raise RuntimeError(
                f"""
                ❌ COLUMNA INEXISTENTE

                Regla:
                {rule.regla_id}

                Tabla:
                {rule.tabla_objetivo}

                Columna:
                {rule.columna}
                """
            )


        filas_ok = (

            df_target

            .select(
                rule.columna
            )

            .distinct()

            .count()
        )


    # ========================================================
    # TIPO NO SOPORTADO
    # ========================================================

    else:

        raise RuntimeError(
            f"""
            ❌ TIPO DE REGLA NO SOPORTADO

            Regla:
            {rule.regla_id}

            Tipo:
            {tipo}
            """
        )


    # ========================================================
    # MÉTRICAS
    # ========================================================

    filas_error = (
        total
        -
        filas_ok
    )


    pct_cumplimiento = (

        round(
            (
                100.0
                *
                filas_ok
                /
                total
            ),
            2
        )

        if total > 0

        else 0.0
    )


    paso = (
        pct_cumplimiento
        >=
        rule.umbral_pct
    )


    # ========================================================
    # ESTADO
    # ========================================================

    if not paso:

        estado = "FAIL"

    elif filas_error > 0:

        estado = "WARNING"

    else:

        estado = "PASS"


    # ========================================================
    # RESULTADO HISTÓRICO
    # ========================================================

    results.append(
        (
            DQ_EXECUTION_ID,
            DQ_EXECUTION_TS,
            rule.regla_id,
            rule.capa,
            rule.tabla_objetivo,
            rule.columna,
            rule.dimension,
            rule.tipo_regla,
            rule.criticidad,
            total,
            filas_ok,
            filas_error,
            pct_cumplimiento,
            rule.umbral_pct,
            paso
        )
    )


    icon = {
        "PASS": "✔",
        "WARNING": "⚠",
        "FAIL": "❌"
    }[
        estado
    ]


    print(
        f"{icon} "
        f"{estado} "
        f"| Total={total:,} "
        f"| OK={filas_ok:,} "
        f"| Error={filas_error:,} "
        f"| Cumplimiento={pct_cumplimiento}% "
        f"| Umbral={rule.umbral_pct}%"
    )


# COMMAND ----------

 # 5 · Construcción del resultado

 Cada regla genera exactamente un registro histórico.

 El contrato persistido incluye:

 - identificador y timestamp de ejecución;
 - regla;
 - capa;
 - tabla;
 - columna;
 - dimensión;
 - tipo de regla;
 - criticidad;
 - métricas;
 - umbral;
 - resultado booleano.

 Para esta versión se esperan exactamente 22 resultados.

In [0]:
# COMMAND ----------

# ============================================================
# 17. CREAR DATAFRAME DE RESULTADOS
# ============================================================

RESULT_SCHEMA = """
    ejecucion_id STRING,
    ejecucion_ts TIMESTAMP,
    regla_id STRING,
    capa STRING,
    tabla_objetivo STRING,
    columna STRING,
    dimension STRING,
    tipo_regla STRING,
    criticidad STRING,
    filas_totales BIGINT,
    filas_ok BIGINT,
    filas_error BIGINT,
    pct_cumplimiento DOUBLE,
    umbral_pct DOUBLE,
    paso BOOLEAN
"""


df_results = (
    spark.createDataFrame(
        results,
        RESULT_SCHEMA
    )
)


result_count = (
    df_results.count()
)


if (
    result_count
    !=
    EXPECTED_RULE_COUNT
):

    raise RuntimeError(
        f"""
        ❌ RESULTADOS DQ SILVER INCOMPLETOS

        Esperados:
        {EXPECTED_RULE_COUNT}

        Generados:
        {result_count}
        """
    )


print(
    f"✔ Resultados generados: "
    f"{result_count}"
)

In [0]:

# COMMAND ----------

# ============================================================
# 18. PERSISTIR RESULTADOS
# ============================================================

RESULT_COLUMNS = [
    "ejecucion_id",
    "ejecucion_ts",
    "regla_id",
    "capa",
    "tabla_objetivo",
    "columna",
    "dimension",
    "tipo_regla",
    "criticidad",
    "filas_totales",
    "filas_ok",
    "filas_error",
    "pct_cumplimiento",
    "umbral_pct",
    "paso"
]


(
    df_results

    .select(
        *RESULT_COLUMNS
    )

    .write

    .format(
        "delta"
    )

    .mode(
        "append"
    )

    .saveAsTable(
        DQ_RESULT_TABLE
    )
)


print(
    f"✔ Resultados DQ Silver "
    f"persistidos en: "
    f"{DQ_RESULT_TABLE}"
)


print(
    f"✔ Execution ID: "
    f"{DQ_EXECUTION_ID}"
)


# COMMAND ----------


 # 6 · Estados de Data Quality

 ## PASS

 La regla alcanza el umbral definido y no presenta registros
 incumplidos.

 ## WARNING

 Existen registros que incumplen la regla, pero el porcentaje
 global todavía alcanza el umbral configurado.

 Un `WARNING` permite continuar y queda registrado para seguimiento.

 ## FAIL

 El porcentaje de cumplimiento se encuentra por debajo del umbral.

 Un `FAIL` solamente bloquea el pipeline cuando la regla posee
 criticidad **alta**.

In [0]:
# COMMAND ----------

# ============================================================
# 19. DERIVAR ESTADO
# ============================================================

df_result_detail = (

    df_results

    .withColumn(

        "estado",

        F.when(
            ~F.col(
                "paso"
            ),

            F.lit(
                "FAIL"
            )
        )

        .when(
            F.col(
                "filas_error"
            )
            >
            0,

            F.lit(
                "WARNING"
            )
        )

        .otherwise(
            F.lit(
                "PASS"
            )
        )
    )
)

In [0]:
# COMMAND ----------

# ============================================================
# 20. DETALLE PASS / WARNING / FAIL
# ============================================================

display(
    df_result_detail

    .orderBy(

        F.when(
            F.col(
                "estado"
            )
            ==
            "FAIL",
            1
        )

        .when(
            F.col(
                "estado"
            )
            ==
            "WARNING",
            2
        )

        .otherwise(
            3
        ),

        F.col(
            "regla_id"
        )
    )
)


 # 7 · Scorecard por dimensión

 El scorecard resume el comportamiento de las reglas agrupadas
 por dimensión de Data Quality.

 Se muestran:

 - cantidad de reglas;
 - reglas `PASS`;
 - reglas `WARNING`;
 - reglas `FAIL`;
 - cumplimiento promedio.

 El promedio permite observar tendencias, pero no reemplaza
 el análisis individual de las reglas críticas.


In [0]:

# COMMAND ----------

# ============================================================
# 21. SCORECARD POR DIMENSIÓN
# ============================================================

df_scorecard = (

    df_result_detail

    .groupBy(
        "dimension"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "reglas"
        ),

        F.sum(
            F.when(
                F.col(
                    "estado"
                )
                ==
                "PASS",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "pass"
        ),

        F.sum(
            F.when(
                F.col(
                    "estado"
                )
                ==
                "WARNING",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "warning"
        ),

        F.sum(
            F.when(
                F.col(
                    "estado"
                )
                ==
                "FAIL",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "fail"
        ),

        F.round(
            F.avg(
                "pct_cumplimiento"
            ),
            2
        ).alias(
            "cumplimiento_promedio"
        )
    )

    .orderBy(
        "dimension"
    )
)


display(
    df_scorecard
)


# COMMAND ----------

 # 8 · Cobertura Review → User

 La regla `DQ-SL-012` mide la cobertura:

 `Review.user_id → User.user_id`

 El archivo User utilizado por YelpConnect corresponde a una
 muestra parcial.

 Por esta razón pueden existir Reviews cuyo `user_id` no aparezca
 en `slv_user`.

 Esta situación:

 - se registra como problema de cobertura;
 - no invalida automáticamente la Review;
 - no representa evidencia de fraude;
 - no constituye por sí sola una señal Trust & Safety.

 La regla permite cuantificar y monitorear la cobertura real
 disponible entre ambos datasets.


In [0]:

# COMMAND ----------

# ============================================================
# 22. REVISAR DQ-SL-012
# ============================================================

display(
    df_result_detail

    .filter(
        F.col(
            "regla_id"
        )
        ==
        "DQ-SL-012"
    )

    .select(
        "regla_id",
        "tabla_objetivo",
        "columna",
        "dimension",
        "criticidad",
        "filas_totales",
        "filas_ok",
        "filas_error",
        "pct_cumplimiento",
        "umbral_pct",
        "estado"
    )
)


# COMMAND ----------


 # 9 · Resumen general

 La ejecución consolida el total de reglas evaluadas y su
 distribución entre:

 - PASS;
 - WARNING;
 - FAIL.

 El resultado global no se determina únicamente por la cantidad
 de fallas: el Quality Gate considera también la criticidad.


In [0]:
# COMMAND ----------

# ============================================================
# 23. RESUMEN GENERAL
# ============================================================

summary_row = (

    df_result_detail

    .agg(

        F.count(
            "*"
        ).alias(
            "total_rules"
        ),

        F.sum(
            F.when(
                F.col(
                    "estado"
                )
                ==
                "PASS",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "pass_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "estado"
                )
                ==
                "WARNING",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "warning_count"
        ),

        F.sum(
            F.when(
                F.col(
                    "estado"
                )
                ==
                "FAIL",
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "fail_count"
        )
    )

    .first()
)


print()
print("=" * 72)
print("RESUMEN DATA QUALITY SILVER")
print("=" * 72)

print(
    f"Reglas ejecutadas : "
    f"{summary_row.total_rules}"
)

print(
    f"PASS              : "
    f"{summary_row.pass_count}"
)

print(
    f"WARNING           : "
    f"{summary_row.warning_count}"
)

print(
    f"FAIL              : "
    f"{summary_row.fail_count}"
)

print("=" * 72)


# COMMAND ----------


 # 10 · Quality Gate Silver

 El Quality Gate determina si Silver puede avanzar hacia
 Trust & Safety.

 ## Bloqueo

 El pipeline se detiene únicamente cuando una regla:

 - tiene criticidad `alta`; y
 - obtiene estado `FAIL`.

 ## No bloqueo

 Los siguientes resultados permanecen registrados, pero no
 detienen automáticamente el pipeline:

 - `WARNING`;
 - `FAIL` de criticidad media;
 - `FAIL` de criticidad baja.

 Esta política permite distinguir defectos críticos de calidad
 de observaciones que requieren seguimiento pero no impiden
 continuar con el procesamiento.

In [0]:
# COMMAND ----------

# ============================================================
# 24. CALCULAR FALLAS CRÍTICAS
# ============================================================

critical_failures = (

    df_result_detail

    .filter(

        (
            F.lower(
                F.trim(
                    F.col(
                        "criticidad"
                    )
                )
            )
            ==
            "alta"
        )

        &

        (
            F.col(
                "estado"
            )
            ==
            "FAIL"
        )
    )

    .count()
)


quality_gate = (

    "FAIL"

    if critical_failures > 0

    else "PASS"
)


print(
    f"Fallas críticas : "
    f"{critical_failures}"
)

print(
    f"Quality Gate    : "
    f"{quality_gate}"
)


In [0]:
# COMMAND ----------

# ============================================================
# 25. MOSTRAR FALLAS CRÍTICAS
# ============================================================

if critical_failures > 0:

    display(

        df_result_detail

        .filter(

            (
                F.lower(
                    F.trim(
                        F.col(
                            "criticidad"
                        )
                    )
                )
                ==
                "alta"
            )

            &

            (
                F.col(
                    "estado"
                )
                ==
                "FAIL"
            )
        )

        .orderBy(
            "regla_id"
        )
    )


# COMMAND ----------

 # Definition of Done — DQ Silver

 - [x] Schema Silver validado.
 - [x] Contrato `dq_reglas` validado.
 - [x] Contrato `dq_resultados` validado.
 - [x] 22 reglas `DQ-SL-*` recuperadas desde gobierno.
 - [x] IDs canónicos validados.
 - [x] Duplicidad de `regla_id` validada.
 - [x] Tipos de regla validados.
 - [x] Criticidades validadas.
 - [x] Umbrales validados.
 - [x] Tablas Silver requeridas verificadas.
 - [x] Reglas ejecutadas.
 - [x] Resultados históricos persistidos.
 - [x] `columna` persistida.
 - [x] `tipo_regla` persistido.
 - [x] PASS / WARNING / FAIL calculado.
 - [x] Scorecard por dimensión generado.
 - [x] Cobertura Review → User evaluada.
 - [x] Quality Gate Silver ejecutado.
 - [x] Ningún dato Silver fue modificado.

 ---

 ## Siguiente etapa

 Si el Quality Gate resulta `PASS`, el pipeline puede continuar
 hacia la construcción de features para Trust & Safety:

 `E7_pipeline_lakehouse/03_construir_features_ts`

In [0]:
# COMMAND ----------

# ============================================================
# 26. RESULTADO PARA ORQUESTADOR
# ============================================================

EXECUTION_FINISHED_AT = (
    datetime.now(
        timezone.utc
    )
)


summary = {

    "status":
        (
            "SUCCESS"
            if quality_gate == "PASS"
            else "FAILED"
        ),

    "pipeline":
        PIPELINE_NAME,

    "execution_id":
        DQ_EXECUTION_ID,

    "layer":
        LAYER,

    "started_at":
        DQ_EXECUTION_TS
        .isoformat(),

    "finished_at":
        EXECUTION_FINISHED_AT
        .isoformat(),

    "rules_evaluated":
        summary_row.total_rules,

    "pass":
        summary_row.pass_count,

    "warning":
        summary_row.warning_count,

    "fail":
        summary_row.fail_count,

    "critical_failures":
        critical_failures,

    "quality_gate":
        quality_gate,

    "result_table":
        DQ_RESULT_TABLE
}


print()
print("=" * 72)
print("DATA QUALITY SILVER FINALIZADO")
print("=" * 72)

print(
    json.dumps(
        summary,
        indent=2
    )
)

print("=" * 72)


# COMMAND ----------

In [0]:
# ============================================================
# 27. APLICAR QUALITY GATE
# ============================================================

if critical_failures > 0:

    raise RuntimeError(
        f"""
        ❌ QUALITY GATE SILVER FALLÓ

        Execution ID:
        {DQ_EXECUTION_ID}

        Reglas críticas fallidas:
        {critical_failures}

        El pipeline NO debe continuar
        hacia Trust & Safety.

        Consultar histórico:

        {DQ_RESULT_TABLE}
        """
    )


print(
    "✔ Quality Gate Silver superado."
)

print(
    "✔ Silver puede continuar "
    "hacia Trust & Safety."
)


dbutils.notebook.exit(
    json.dumps(
        summary
    )
)